In [ ]:
%pip install realspace-tb

In [1]:
import os
import sys
# 1. Get the folder where the notebook is (Folder 2)
notebook_dir = os.getcwd()

# 2. Go up one level and then into Folder 1
realspace_tb_path = os.path.abspath(os.path.join(notebook_dir, '..'))

# 3. Add to sys.path so you can import
if realspace_tb_path not in sys.path:
    sys.path.append(realspace_tb_path)

import realspace_tb as tb
import realspace_tb.orbitronics_2d as orb
import numpy as np
import matplotlib

if not hasattr(matplotlib.RcParams, '_get'):
    matplotlib.RcParams._get = lambda self, key: self.get(key)

import matplotlib.pyplot as plt
from pathlib import Path
import pickle

# Armchair ribbon

In [2]:
Lx,Ly = 17, 10
field_amplitude=orb.RampedACFieldAmplitude(E0=5e-3, omega=2.0, T_ramp=40, direction=np.array([0, 1]))
geo_ac = orb.HoneycombLatticeGeometry(
    Lx=Lx, 
    Ly=Ly, 
    pbc_x=False, 
    pbc_y=True,
)


H_ac = orb.LinearFieldHamiltonianPeierls(geo_ac, field_amplitude)

In [ ]:
rho = H_ac.ground_state_density_matrix(fermi_level=0.0)
animation = orb.observables.LatticeFrameObservable(H_ac.geometry, window=tb.MeasurementWindow(stride=15, start_time=field_amplitude.T_ramp), hamiltonian=H_ac)
orb_pol = orb.observables.OrbitalPolarizationObservable(H_ac.geometry, window=tb.MeasurementWindow(stride=15, start_time=field_amplitude.T_ramp), hamiltonian=H_ac)
tb.RK4NeumannSolver().evolve(rho, H_ac, dt=0.01, total_time=field_amplitude.T_ramp + 40, observables=[animation, orb_pol], tau=float("inf"))

In [ ]:
electric_field_vectors = [field_amplitude.at_time(t) * field_amplitude.direction for t in animation.measurement_times]
orb.save_simulation_animation(animation, "zz.mp4", 
                              electric_field_vectors=electric_field_vectors,
                              field_arrow_label="$\\vec E(t)$",
                              field_arrow_color="#14B8A6",
                              )

In [ ]:
# not for 29, but 27 and 31

# for 12, 11
# not for strong damping
# Ly dependent?!
# only for 1.98 < omega and annoying horizintal modes appear at 2.5. So

In [ ]:
# plot the currents: this shows how the currents are stronger in the center
max_orb_pol = np.argmax(orb_pol.values[:, 0])
plt.plot(np.abs(animation.values["currents"][max_orb_pol-1]))

In [ ]:
print(f"Maximum orbital polarization at time {orb_pol.measurement_times[max_orb_pol]:.2f} with value {orb_pol.values[max_orb_pol, 0]:.4f}")

orb.show_simulation_frame(
    animation,
    0,
    show_oam_direction_arrows=True,
    frame_texts=[""]* len(animation.measurement_times),
    show=False, # to use savefig
)

#plt.savefig("ac_max_orb_pol.pdf", bbox_inches="tight")

# Zigzag Nanoribbon

In [ ]:
field_amplitude=orb.RampedACFieldAmplitude(E0=1e-3, omega=0.5, T_ramp=40, direction=np.array([1, 0]))
geo_zz = orb.HoneycombLatticeGeometry(
    Lx=10, 
    Ly=13, 
    pbc_x=True, 
    pbc_y=False,
)

H_zz = orb.LinearFieldHamiltonianPeierls(geo_zz, field_amplitude)
rho = H_zz.ground_state_density_matrix(fermi_level=0.0)
animation = orb.observables.LatticeFrameObservable(H_zz.geometry, window=tb.MeasurementWindow(stride=15, start_time=field_amplitude.T_ramp), hamiltonian=H_zz)
orb_pol = orb.observables.OrbitalPolarizationObservable(H_zz.geometry, window=tb.MeasurementWindow(stride=15, start_time=field_amplitude.T_ramp), hamiltonian=H_zz)
tb.RK45Solver().evolve(rho, H_zz, dt=0.01, total_time=field_amplitude.T_ramp + 50, observables=[animation, orb_pol], tau=20)

In [ ]:
max_orb_pol = np.argmax(orb_pol.values[:, 1])
print(f"Maximum orbital polarization at time {orb_pol.measurement_times[max_orb_pol]:.2f} with value {orb_pol.values[max_orb_pol, 1]:.4f}")

orb.show_simulation_frame(
    animation,
    max_orb_pol+15,
    show_oam_direction_arrows=True,
    frame_texts=[""]* len(animation.measurement_times),
    oam_arrow_threshold=0.0001,
    show=False, # to use savefig
)

plt.savefig("zz_max_orb_pol.pdf", bbox_inches="tight")

In [ ]:
electric_field_vectors = [field_amplitude.at_time(t) * field_amplitude.direction for t in animation.measurement_times]
orb.save_simulation_animation(animation, "zz.mp4", 
                              electric_field_vectors=electric_field_vectors,
                              field_arrow_label="$\\vec E(t)$",
                              field_arrow_color="#14B8A6",
                              field_arrow_type="horizontal",
                              )

In [ ]:
def save_findings(animation, zz_ribbon, dirname):
    # make a directory at dirname
    findings_dir = Path(dirname)
    findings_dir.mkdir(exist_ok=False)

    pickle_path = findings_dir / "geometry.pickle"
    with pickle_path.open("b+w") as f:
        pickle.dump(zz_ribbon, f)

    markdown_path = findings_dir / "stats.md"
    with markdown_path.open("w") as md:
        md.writelines([f"---\n{key}: {value}\n\n" for key, value in zz_ribbon.field_amplitude.__dict__.items()])
        md.writelines([f"---\n{key}: {value}\n\n" for key, value in zz_ribbon.__dict__.items()])
        
    anim_path = findings_dir / "animation.mp4"
    orb.save_simulation_animation(animation, anim_path)

In [ ]:
save_findings(animation, H_ac, "niceSeparationVanHove3")